# FADE · Select instead of replace — **CPU only, ~3 min**

Set the accelerator to **None**. Nothing here generates text; it re-scores traces
that already exist.

The stage-3 run replaced the answer on every flagged problem. Its own numbers say
that was the wrong call 50 times out of 108 discordant pairs:

```
pass 1            367/1319 = 27.8%
final (REPLACE)   375/1319 = 28.4%   +0.6%   p = 0.50
gained 58, lost 50
```

This notebook keeps the same two traces and changes only **which one is kept**,
using three gold-free signals in cascade:

1. **the answers agree** — nothing to choose
2. **arithmetic validity** — prefer the trace whose stated equations actually hold.
   On paired train data this picks correctly 83.6% of the time.
3. **the detector** — otherwise keep whichever trace scores lower `p_wrong`

No generation. No gold in the decision; gold is opened only to score, at the end.

**Sidebar:** accelerator None · Internet on · Secret `GH_TOKEN`.

### 1 · Setup

In [ ]:
import os, sys, shutil, subprocess, json, collections
!pip -q install -U "transformers>=4.40" sentence-transformers sympy scikit-learn datasets scipy joblib 2>/dev/null | tail -1
REPO='/kaggle/working/fade'; STORES='/kaggle/working/stores'; os.makedirs(STORES,exist_ok=True)
if os.path.exists(REPO): shutil.rmtree(REPO)
!git clone --depth 1 https://github.com/S2V3/fade.git {REPO}
sys.path.insert(0,REPO)
import config
VER=config.FADE_CODE_VERSION; TAG=VER.replace('-','')
from kaggle_secrets import UserSecretsClient
GH=UserSecretsClient().get_secret('GH_TOKEN')
print('code',VER,'OK')

### 2 · Fetch the stage-3 store and the classifier

Both are on the results branch. The store holds both traces per flagged
problem; the classifier is only the third tier of the cascade.

In [ ]:
AUTH=f'https://{GH}@github.com/S2V3/fade.git'; BRANCH='results'
def fetch(prefix, dest, need=None):
    w=f'/kaggle/working/_rs_{abs(hash(prefix))%9999}'; shutil.rmtree(w,ignore_errors=True)
    if subprocess.run(['git','clone','--depth','1','--branch',BRANCH,AUTH,w],
                      capture_output=True,text=True).returncode: return False
    cands=[]
    for d in sorted(os.listdir(f'{w}/results')):
        if not d.startswith(prefix): continue
        full=f'{w}/results/{d}'
        if not os.path.isdir(full): continue
        if need and not os.path.exists(f'{full}/{need}'): continue
        size = (os.path.getsize(f'{full}/{need}') if need
                else sum(1 for _ in open(f'{full}/results.jsonl')))
        cands.append((size, d, full))
    if not cands:
        print(f'  nothing matching {prefix}*'); return False
    cands.sort(); shutil.rmtree(dest,ignore_errors=True); shutil.copytree(cands[-1][2],dest)
    print(f'  restored {cands[-1][1]}')
    return True

STORE=f'{STORES}/store_ctctest'; CTC_DIR=f'{STORES}/ctc2'
assert fetch(f'store_ctctest_{TAG}_a1', STORE), 'stage-3 store not on the branch'
assert fetch(f'ctc2_{TAG}', CTC_DIR, need='ctc2.joblib'), 'ctc2.joblib not on the branch'
rows=[json.loads(l) for l in open(f'{STORE}/results.jsonl') if l.strip()]
print(f'\n  {len(rows)} rows, {sum(1 for r in rows if r.get("retry_trace"))} with a retry')

### 3 · Decide which trace to keep — gold-free

Nothing here reads a reference answer. The picks depend only on the
question, the two traces, and the classifier.

In [ ]:
import joblib, warnings; warnings.filterwarnings('ignore')
from extraction import extract_equations, trace_body
m=joblib.load(f'{CTC_DIR}/ctc2.joblib')

# fraction of the trace's stated equations that actually hold
def truth(t):
    e=extract_equations(trace_body(t or ''))
    return None if not e else sum(x.is_true for x in e)/len(e)

picks={}
for r in rows:
    if not r.get('retry_trace'): continue
    q,a,b = r['question'], r['trace'], r['retry_trace']
    if str(r.get('final_answer'))==str(r.get('retry_answer')):
        picks[r['id']]=('retry','agree')
        continue
    ta,tb=truth(a),truth(b)
    if ta is not None and tb is not None and ta!=tb:
        picks[r['id']]=(('pass1' if ta>tb else 'retry'),'arith')
    else:
        pa=m.predict(q,a)['p_wrong']; pb=m.predict(q,b)['p_wrong']
        picks[r['id']]=(('pass1' if pa<pb else 'retry'),'detector')

print('  decided by:', dict(collections.Counter(v[1] for v in picks.values())))
kp=sum(1 for v in picks.values() if v[0]=='pass1')
print(f'  keeps pass-1 on {kp}/{len(picks)} ({kp/len(picks):.0%}) -- replace-always keeps 0')

### 4 · Score both policies — the only place gold is opened

Same traces, same detector, same budget. The one thing that differs is
which of the two answers is kept.

In [ ]:
import kaggle_run as K
K.SPLIT_TAG='test'; K.MODEL_ID='meta-llama/Llama-2-7b-chat-hf'
gold={p['question']:p for p in K._load_split('test',None)}
assert not [k for r in rows for k in ('gold_answer','correct') if k in r], 'run was not gold-free'

def ok(q,t):
    g=gold[q]
    return bool(K.score_trace(q,t,g['answer'],g['gold_answer'])[0].correct)

p1=repl=sel=0; gr=lr=gs=ls=0
tally=collections.Counter(); A={}; Br={}; Bs={}
for r in rows:
    q=r['question']
    if q not in gold: continue
    c1=ok(q,r['trace']); p1+=c1; A[r['id']]=c1
    if not r.get('retry_trace'):
        repl+=c1; sel+=c1; Br[r['id']]=c1; Bs[r['id']]=c1; continue
    c2=ok(q,r['retry_trace']); repl+=c2; Br[r['id']]=c2
    cs = c1 if picks[r['id']][0]=='pass1' else c2
    sel+=cs; Bs[r['id']]=cs
    if c1!=c2:
        gr+=(not c1) and c2; lr+=c1 and (not c2)
        gs+=(not c1) and cs; ls+=c1 and (not cs)
        tally[(picks[r['id']][1], 'right' if cs==max(c1,c2) else 'wrong')]+=1

n=len(rows)
print('='*60)
print(f'  pass 1              {p1:>5}/{n} = {p1/n:>6.1%}')
print(f'  REPLACE always      {repl:>5}/{n} = {repl/n:>6.1%}   ({(repl-p1)/n:+.1%})')
print(f'  SELECT  (gold-free) {sel:>5}/{n} = {sel/n:>6.1%}   ({(sel-p1)/n:+.1%})')
print(f'\n  on discordant pairs')
print(f'    replace   gained {gr:>3}  lost {lr:>3}   net {gr-lr:+d}')
print(f'    select    gained {gs:>3}  lost {ls:>3}   net {gs-ls:+d}')
print(f'\n  which signal decided, and was it right')
for w in ('agree','arith','detector'):
    a_,b_=tally[(w,'right')],tally[(w,'wrong')]
    if a_+b_: print(f'    {w:<10}{a_:>4} right {b_:>4} wrong   {a_/(a_+b_):>6.1%}')
from stats import mcnemar
for lbl,B in (('REPLACE',Br),('SELECT',Bs)):
    mc=mcnemar(A,B)
    print(f'  {lbl:<8} McNemar p = {mc["p"]:.2e}'+('  SIGNIFICANT' if mc['p']<0.05 else ''))

### How to read it

**SELECT beats REPLACE at p < 0.05** — the claim becomes *a second attempt is only
worth taking if you can tell which attempt was better, and that judgement can be
made symbolically*. Report both rows; the gap between them is the contribution.

**SELECT ≈ REPLACE** — the selector is no better than chance on discordant pairs.
Report the break-even analysis instead: detection generalises, repair cannot pay at
12% recovery against 75% damage.

**`arith` well above 50% while `detector` sits near it** — the arithmetic check
carries the signal and the trained classifier does not. Say that explicitly; a
symbolic check beating a learned model at the same job is the more interesting
result.